# Notebook 1: Compilation & Static QASM Analysis
**Paper 5 – Cross-Framework Quantum Algorithm Benchmarking**

**Purpose**: Run the full compilation pipeline (all algorithms × 3 frameworks → OpenQASM 3.0)
and perform structural analysis on the generated QASM files.

**Outputs**:
- `benchmarks/qasm_outputs/*.qasm` — compiled QASM files
- `benchmarks/metrics/compilation_times.csv`
- `benchmarks/metrics/structural_metrics.csv`

**Pipeline step**: Step 1 of 5

In [ ]:
# Setup — run from QCanvas project root
import os, sys
QCANVAS_ROOT = os.path.abspath('../..')
if QCANVAS_ROOT not in sys.path:
    sys.path.insert(0, QCANVAS_ROOT)
print('QCanvas root:', QCANVAS_ROOT)

In [ ]:
# ── Phase 1: Compile all algorithms ─────────────────────────────────────────
import sys
sys.path.insert(0, '../../')
from benchmarks.scripts.compile_all import run_all_compilations, ALGORITHM_REGISTRY

print(f'Algorithm registry: {len(ALGORITHM_REGISTRY)} algorithms')
total_combos = sum(len(e["qubits"]) * 3 for e in ALGORITHM_REGISTRY)
print(f'Total (algo × qubits × framework) combinations: {total_combos}')
print('\nStarting compilation...')

df_times = run_all_compilations(n_repeats=10, verbose=True)
print(f'\nCompilation complete. {df_times["success"].sum()}/{len(df_times)} succeeded.')

In [ ]:
# ── Compilation timing summary ───────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
from benchmarks.scripts.figure_styles import apply_paper_style, FRAMEWORK_COLORS, FRAMEWORK_LABELS

df_times = pd.read_csv('benchmarks/metrics/compilation_times.csv')
success = df_times[df_times['success']]

print('Mean compilation time per framework (ms):')
print(success.groupby('framework')['mean_compile_ms'].agg(['mean','std']).round(3))

In [ ]:
# ── Phase 2: Static QASM analysis ───────────────────────────────────────────
from benchmarks.scripts.analyze_qasm import run_static_analysis

df_struct = run_static_analysis('benchmarks/qasm_outputs')
df_struct.to_csv('benchmarks/metrics/structural_metrics.csv', index=False)
print(f'Structural metrics: {len(df_struct)} rows')
df_struct.head()

In [ ]:
# ── Fig. 2: Gate count per algorithm per framework ───────────────────────────
apply_paper_style()
from benchmarks.scripts.figure_styles import plot_grouped_bar, save_figure

# Use fixed qubit count per algorithm for cross-framework comparison
df_fixed = df_struct[df_struct['n_qubits'] == df_struct.groupby('algorithm')['n_qubits'].transform('min')]

fig, ax = plt.subplots(figsize=(12, 5))
plot_grouped_bar(
    ax, df_fixed,
    x_col='algorithm', y_col='total_gates',
    title='Fig. 2 — Total Gate Count per Algorithm and Framework',
    ylabel='Total Gates',
)
plt.tight_layout()
save_figure(fig, 'fig02_gate_counts', 'structural')
plt.show()

In [ ]:
# ── Fig. 3: Circuit depth per algorithm per framework ────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
plot_grouped_bar(
    ax, df_fixed,
    x_col='algorithm', y_col='circuit_depth',
    title='Fig. 3 — Circuit Depth per Algorithm and Framework',
    ylabel='Circuit Depth',
)
plt.tight_layout()
save_figure(fig, 'fig03_circuit_depth', 'structural')
plt.show()

In [ ]:
# ── Fig. 4: Gate composition (stacked bar) ───────────────────────────────────
import numpy as np

gate_cols = [c for c in df_struct.columns if c.startswith('gate_') and c != 'gate_MEASURE']
fw_totals = {}
for fw in ['qiskit', 'cirq', 'pennylane']:
    sub = df_fixed[df_fixed['framework'] == fw]
    fw_totals[FRAMEWORK_LABELS[fw]] = sub[gate_cols].sum()

df_gate_comp = pd.DataFrame(fw_totals).T
df_gate_comp.columns = [c.replace('gate_', '') for c in df_gate_comp.columns]

# Keep top 6 gate types by total count
top_gates = df_gate_comp.sum().nlargest(6).index
df_plot = df_gate_comp[top_gates]

apply_paper_style()
fig, ax = plt.subplots(figsize=(10, 5))
df_plot.plot(kind='bar', stacked=True, ax=ax,
             color=plt.cm.Set2(np.linspace(0, 1, len(top_gates))))
ax.set_title('Fig. 4 — Gate Type Composition by Framework')
ax.set_ylabel('Total Gate Count')
ax.set_xlabel('Framework')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
save_figure(fig, 'fig04_gate_composition', 'structural')
plt.show()

In [ ]:
# ── QASM 3.0 modifier usage summary ────────────────────────────────────────
print('QASM 3.0 Modifier Usage (ctrl @, inv @, pow()):' )
mod_summary = df_struct.groupby('framework')[['ctrl_count', 'inv_count', 'pow_count']].sum()
print(mod_summary.to_string())

print('\nThis demonstrates QASM 3.0-specific features that distinguish QCanvas')
print('output from traditional QASM 2.0 transpilers.')